<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 메모리 효율적인 모델 가중치 로딩 (Memory-efficient Model Weight Loading)

- 이 노트북은 GPU (또는 CPU) 메모리가 제한된 상황에서 더 큰 사전훈련된 또는 미세조정된 모델을 로딩하는 팁을 제공합니다
- 특히, `torch.save(model.state_dict(), "model.pth")`를 사용하여 모델을 저장한 경우(예를 들어, 5-7장에서)와 새로운 세션에서 나중에 지속적인 사전훈련이나 추가 미세조정을 위해 로딩하고자 하는 경우에 초점을 맞춥니다
- 예제에서는 LLM을 사용하지만, 이 노트북에서 설명하는 방법들은 일반적이며 LLM뿐만 아니라 모든 PyTorch 모델 로딩에 적용됩니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

In [ ]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
## 1. 벤치마크 유틸리티 (Benchmark utilities)

- 먼저 VRAM (GPU 메모리)을 추적하는 유틸리티 코드를 정의해보겠습니다
- 나중에 메인 시스템 RAM (CPU 메모리)을 추적하는 도구도 소개할 것입니다
- 이러한 함수들의 목적은 나중에 적용할 때 명확해질 것입니다

In [ ]:
import gc
import time
import torch


def start_memory_tracking():
    """GPU 메모리 추적을 초기화합니다."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("이 노트북은 CUDA GPU를 위해 작성되었지만 CUDA를 사용할 수 없습니다.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # 바이트를 GB로 변환
    print(f"최대 GPU 메모리 할당량: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # 메모리가 정리될 수 있도록 버퍼 시간 제공
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"최대 GPU 메모리 할당량: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. 모델 설정 (Model setup)

- 이 코드 섹션은 모델 자체를 설정합니다
- 여기서는 더 흥미롭게 만들기 위해 "large" GPT-2 모델을 사용합니다 (이 노트북의 메모리 요구사항과 실행 시간을 줄이려면 "gpt2-small (124M)"을 사용할 수 있습니다)

In [ ]:
from previous_chapters import GPTModel
# `previous_chapters.py` 파일이 로컬에서 사용할 수 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은 https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg 를 참조하세요
# 예를 들어,
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기
    "context_length": 1024,  # 컨텍스트 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # Query-key-value 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- 이제 GPU 메모리 함수들이 실제로 작동하는 것을 살펴보겠습니다:

In [ ]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

- 추가적으로, 예제 텐서를 전달하여 모델이 정상적으로 작동하는지 확인해보겠습니다

In [ ]:
# 모델이 작동하는지 테스트 (여기서는 메모리를 추적할 필요 없음)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- 다음으로, 모델을 사전훈련하고 나중에 사용하기 위해 저장한다고 가정해보겠습니다
- 여기서는 간단함을 위해 실제 사전훈련을 건너뛰고 초기화된 모델만 저장합니다 (하지만 동일한 개념이 적용됩니다)

In [ ]:
# 훈련 코드가 여기에 들어갑니다...

model.train()
torch.save(model.state_dict(), "model.pth")

- 마지막으로, Python 세션에서 모델과 예제 텐서를 삭제하여 GPU 메모리를 재설정합니다

In [ ]:
del model, test_input
cleanup()

&nbsp;
## 3. 가중치 로딩 (Weight loading)

- 이제 사전훈련된 모델 가중치를 로딩하는 흥미로운 부분이 시작됩니다
- 이전에 저장된 모델을 로딩하는 데 얼마나 많은 GPU 메모리가 필요한지 살펴보겠습니다

In [ ]:
# 그다음 사전훈련된 가중치를 로딩합니다

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

- 메모리가 이전 세션보다 2배 크다는 것을 주목하세요
- 이는 짧은 시간 동안 메모리에 같은 모델이 두 번 존재하기 때문입니다:
  - 첫 번째는 `model.to(device)`를 통해
  - 두 번째는 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))` 코드 라인을 통해; 결국 로딩된 모델 가중치가 모델에 복사되고 `state_dict`가 폐기되지만, 짧은 시간 동안 메인 모델과 로딩된 `state_dict` 둘 다 메모리에 존재합니다
- 남은 섹션들은 이 문제를 해결하는 데 초점을 맞춥니다
- 하지만 먼저 모델을 테스트하고 GPU 메모리를 재설정해보겠습니다


In [ ]:
# 모델이 작동하는지 테스트 (여기서는 메모리를 추적할 필요 없음)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

&nbsp;
## 4. 순차적으로 가중치 로딩 (Loading weights sequentially)

- 이전 섹션에서 강조된 GPU 메모리에 모델 가중치가 두 번 존재하는 문제에 대한 하나의 해결책은 모델을 순차적으로 로딩하는 것입니다
- 아래에서 우리는:
  - 먼저 모델을 GPU 메모리에 로딩합니다
  - 그다음 모델 가중치를 CPU 메모리에 로딩합니다
  - 그리고 마지막으로 각 파라미터를 하나씩 GPU 메모리로 복사합니다


In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# 가중치를 모델의 파라미터에 순차적으로 복사
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"경고: {name}이 state_dict에서 찾을 수 없습니다.")

print_memory_usage()

- 위에서 볼 수 있듯이, 메모리 사용량이 이전보다 훨씬 낮습니다
- 메모리가 6.4에서 6.7 GB로 증가하는 것을 주목하세요. 이는 처음에는 모델만 메모리에 있고, 그다음에는 모델과 1개의 파라미터 텐서가 메모리에 있기 때문입니다 (파라미터 텐서를 모델에 할당하기 위해 ".to"를 사용하여 일시적으로 GPU로 이동합니다)
- 전체적으로, 이는 상당한 개선입니다
- 다시, 모델을 간단히 테스트하고 다음 섹션을 위해 GPU 메모리를 재설정하겠습니다

In [ ]:
# 모델이 작동하는지 테스트 (여기서는 메모리를 추적할 필요 없음)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

&nbsp;
## 5. 낮은 CPU 메모리로 모델 로딩 (Loading the model with low CPU memory)

- 이전 섹션에서, 가중치(`state_dict`)를 먼저 CPU 메모리에 로딩한 다음 하나씩 모델로 복사하여 GPU 메모리 사용을 줄였습니다
- 하지만 CPU 메모리가 제한된 경우에는 어떻게 해야 할까요?
- 이 섹션에서는 PyTorch의 소위 `"meta"` 디바이스 접근법을 사용하여 큰 GPU 메모리를 가지지만 작은 CPU 메모리를 가진 머신에서 모델을 로딩합니다
- 하지만 먼저, CPU 메모리를 모니터링하는 편의 함수를 정의해보겠습니다

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # 함수를 실행하기 전 기준 메모리 사용량을 측정
    baseline_mem = process.memory_info().rss / 1024 ** 3  # GB 단위

    # 별도 스레드에서 메모리 모니터링 시작
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # GB로 변환
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # 함수 실행
    func(*args, **kwargs)

    # 모니터링 중단
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- 시작하면서, 이전 섹션의 순차적 가중치 로딩 접근법의 CPU 메모리를 추적해보겠습니다

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # 가중치를 모델의 파라미터에 순차적으로 복사
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"경고: {name}이 state_dict에서 찾을 수 없습니다.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> 최대 CPU 메모리 할당량: {peak_memory_used:.1f} GB")

- 이제 낮은 CPU 메모리를 가지지만 큰 GPU 메모리를 가진 머신이 있다고 가정해보겠습니다
- PyTorch의 소위 "meta" 디바이스를 도입하여 CPU 메모리와 GPU 메모리 사용량을 트레이드오프할 수 있습니다
- PyTorch의 meta 디바이스는 데이터에 대한 실제 메모리를 할당하지 않고 텐서를 생성할 수 있게 해주는 특수한 디바이스 타입으로, 효과적으로 "meta" 텐서를 생성합니다
- 이는 메모리 할당의 오버헤드 없이 텐서 모양과 타입이 필요한 모델 분석이나 아키텍처 정의와 같은 작업에 유용합니다

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # 가중치를 모델의 파라미터에 순차적으로 복사
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"경고: {name}이 state_dict에서 찾을 수 없습니다.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> 최대 CPU 메모리 할당량: {peak_memory_used:.1f} GB")

- 위에서 볼 수 있듯이, meta-디바이스에서 모델을 생성하고 가중치를 직접 GPU 메모리로 로딩함으로써 효과적으로 CPU 메모리 요구사항을 줄였습니다
- "그러면 순차적 가중치 로딩이 여전히 필요한가요? 그리고 이것이 원래 접근법과 어떻게 비교되나요?"라고 물을 수 있습니다
- 비교를 위해 간단한 PyTorch 가중치 로딩 접근법(이 노트북의 첫 번째 가중치 로딩 섹션에서)을 확인해보겠습니다:

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> 최대 CPU 메모리 할당량: {peak_memory_used:.1f} GB")

- 위에서 볼 수 있듯이, meta 디바이스 없이 "간단한" 가중치 로딩은 더 많은 메모리를 사용합니다
- 다시 말해, CPU 메모리가 제한된 머신이 있다면, meta 디바이스 접근법을 사용하여 모델 가중치를 직접 GPU 메모리로 로딩하여 최대 CPU 메모리 사용량을 줄일 수 있습니다

&nbsp;
## 6. `mmap=True` 사용 (권장)

- 중급 또는 고급 `torch.load` 사용자라면, 이러한 접근법들이 PyTorch의 `mmap=True` 설정과 어떻게 비교되는지 궁금할 수 있습니다
- PyTorch의 `mmap=True` 설정은 메모리 매핑된 파일 I/O를 가능하게 하여, 텐서가 디스크 저장소에서 직접 데이터에 접근할 수 있게 하므로, RAM이 제한된 경우 전체 파일을 RAM으로 로딩하지 않음으로써 메모리 사용량을 줄입니다
- 또한, [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402)의 도움이 되는 코멘트를 참조하세요
- 처음 보기에는 위의 순차적 접근법들보다 효율성이 떨어져 보일 수 있습니다:

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> 최대 CPU 메모리 할당량: {peak_memory_used:.1f} GB")

- CPU RAM 사용량이 이렇게 높은 이유는 이 머신에서 충분한 CPU RAM이 사용 가능하기 때문입니다
- 하지만 CPU RAM이 제한된 머신에서 실행한다면, `mmap` 접근법은 더 적은 메모리를 사용할 것입니다

&nbsp;
## 7. 기타 방법들 (Other methods)

- 이 노트북은 PyTorch에서 가중치를 로딩하는 간단한 내장 방법들에 초점을 맞춥니다
- CPU 메모리가 제한된 경우에 권장되는 접근법은 위에서 충분히 설명한 `mmap=True` 접근법입니다
- 또는, 각 가중치 텐서를 별도로 저장하고 로딩하는 무차별적인 접근법도 또 다른 옵션입니다:

In [ ]:
model = GPTModel(BASE_CONFIG)
# `model`이 훈련된 모델이라고 가정
state_dict = model.state_dict()

# 개별 파라미터 파일을 저장할 디렉터리 생성
os.makedirs("model_parameters", exist_ok=True)

# 각 파라미터 텐서를 별도로 저장
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # 메모리 해제
            else:
                print(f"경고: {name}을 {param_dir}에서 찾을 수 없습니다.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> 최대 CPU 메모리 할당량: {peak_memory_used:.1f} GB")